In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "google/gemma-4-E4B-it" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        #dtype=torch.bfloat16,
        device_map=device,
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

основной поток 2560 ───────────────────────────────┐
                                                   + → 2560
основной поток 2560 → gate 256                     │
                              × per-layer input 256 │
                              → projection 2560 ────┘

There is another branch of per_layer_input in forward of model.

Builded from special per-layer embeddings. Or from projection of main embedding (???)

In [3]:
# token t:
    # main embedding/hidden state: 2560
    # layer 0 extra input:          256
    # layer 1 extra input:          256
    # ...
    # layer 41 extra input:         256

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [5]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [6]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

### Without MOE block, just mlp

### Learning with TP and ours

In [7]:
from prune_and_train_model_for_comparsion import full_load_prune_learn_pipeline
FRACTION_ATTN_LAYERS = [0, 8, 16, 32, 40]
FRACTION_MLP_LAYERS = [4, 12, 20, 30] 
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2, 99, 125, 157, 250] #q proj
MLP_OUT_FRACTION = [3, 5, 1000, 2000, 3001, 4002, 5003, 6004, 7005, 8006, 9007, 10009] #up proj
FRACTION_SEED = 0

In [8]:
full_load_prune_learn_pipeline(MODEL,
                               load_model,
                               evaluation_blocks,
                               
    FRACTION_ATTN_LAYERS, FRACTION_MLP_LAYERS, ATTN_OUT_FRACTION, MLP_OUT_FRACTION,
    is_torch_pruning=False,
    rescale=False,
    seed=FRACTION_SEED,
    batch_size=1
)

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

fraction prune_task:
  blocks.0.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99, 125, 157, 250], kv_lora_idxs_deepseek=None)
  blocks.8.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99, 125, 157, 250], kv_lora_idxs_deepseek=None)
  blocks.16.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99, 125, 157, 250], kv_lora_idxs_deepseek=None)
  blocks.32.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99, 125, 157, 250], kv_lora_idxs_deepseek=None)
  blocks.40.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99, 125, 157, 250], kv_lora_idxs_deepseek=None)
  blocks.4.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002, 5003, 6004, 7005, 8006, 9007, 10009], kv_lora_idxs_deepseek=None)
  blocks.12.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002, 5003, 6004, 7005, 8006, 9007, 10009], kv_lora_idxs_deepseek=None)
  blocks.20.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002, 5003, 6004, 7005, 8006, 9007, 1000

/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.vision_tower._original_component.encoder.layers.1.post_feedforward_layernorm.weight', 'model.vision_tower._original_component.encoder.layers.7.input_layernorm.weight', 'model.audio_tower.layers.2.feed_forward1.ffw_layer_2.linear.weight', 'model.language_model.layers.17._original_component.self_attn._original_component.q_norm._original_component.weight', 'model.audio_tower.layers.1.self_attn.per_dim_scale', 'model.audio_tower.layers.6.self_attn.k_proj.linear.weight', 'model.audio_tower.layers.8.self_attn.per_dim_scale', 'model.audio_tower.layers.10.self_attn.post.linear.weight', 'model.audio_tower.layers.11.norm_pre_attn.weight', 'model.language_model.layers.16._original_component.post_attention_layernorm._original_component.weight', 'model.language_model.layers.36._original_component.post_attention_layernorm._original_component

removed group indices by module:
  blocks.0.attn.q_proj: count=96, idxs=[1, 2, 29, 99, 122, 125, 129, 130, 157, 227, 250, 253, 257, 258, 285, 355, 378, 381, 385, 386, 413, 483, 506, 509, 513, 514, 541, 611, 634, 637, 641, 642, 669, 739, 762, 765, 769, 770, 797, 867, 890, 893, 897, 898, 925, 995, 1018, 1021, 1025, 1026, 1053, 1123, 1146, 1149, 1153, 1154, 1181, 1251, 1274, 1277, 1281, 1282, 1309, 1379, 1402, 1405, 1409, 1410, 1437, 1507, 1530, 1533, 1537, 1538, 1565, 1635, 1658, 1661, 1665, 1666, 1693, 1763, 1786, 1789, 1793, 1794, 1821, 1891, 1914, 1917, 1921, 1922, 1949, 2019, 2042, 2045]
  blocks.8.attn.q_proj: count=96, idxs=[1, 2, 29, 99, 122, 125, 129, 130, 157, 227, 250, 253, 257, 258, 285, 355, 378, 381, 385, 386, 413, 483, 506, 509, 513, 514, 541, 611, 634, 637, 641, 642, 669, 739, 762, 765, 769, 770, 797, 867, 890, 893, 897, 898, 925, 995, 1018, 1021, 1025, 1026, 1053, 1123, 1146, 1149, 1153, 1154, 1181, 1251, 1274, 1277, 1281, 1282, 1309, 1379, 1402, 1405, 1409, 1410, 1437, 1

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 3 has a total capacity of 39.38 GiB of which 49.81 MiB is free. Process 3308122 has 528.00 MiB memory in use. Process 332061 has 38.80 GiB memory in use. Of the allocated memory 38.17 GiB is allocated by PyTorch, and 134.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)